# go-ai book-state teacher/student demo

This notebook is a UI client for the Go demo service. The model call path stays in `go-ai` and any loaded `go-inference` backend; Python only posts JSON to `/ask`.

Start the Go service first, for example:

```sh
go run ./go/cmd/book-state-demo -mock -title "Meditations" -excerpt-file /Users/snider/Code/lthn/LEM/training/lem/composure/transparency-aurelius-meditations.txt
```

In [ ]:
import json
import os
import urllib.error
import urllib.request

import gradio as gr

DEFAULT_ENDPOINT = os.getenv("CORE_BOOK_STATE_DEMO_URL", "http://127.0.0.1:8787")

In [ ]:
def post_json(endpoint, path, payload):
    request = urllib.request.Request(
        endpoint.rstrip("/") + path,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=300) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8")
        try:
            message = json.loads(detail).get("error", detail)
        except json.JSONDecodeError:
            message = detail
        raise gr.Error(message) from exc


def ask_book_state(endpoint, question, max_tokens, student_uses_state):
    question = question.strip()
    if not question:
        raise gr.Error("Question is required")
    response = post_json(endpoint, "/ask", {
        "question": question,
        "max_tokens": int(max_tokens),
        "student_uses_book_state": bool(student_uses_state),
    })
    messages = [{"role": "user", "content": question}]
    student = response.get("student_answer", "").strip()
    if student:
        messages.append({"role": "assistant", "content": "Student:\n" + student})
    messages.append({"role": "assistant", "content": "Teacher:\n" + response.get("teacher_answer", "")})
    state = response.get("state", {})
    return messages, response, {
        "title": state.get("title"),
        "entry_uri": state.get("entry_uri"),
        "prefix_tokens": state.get("prefix_tokens"),
    }

In [ ]:
with gr.Blocks(title="go-ai Book State Demo") as demo:
    gr.Markdown("# go-ai book state teacher/student")
    with gr.Row():
        endpoint = gr.Textbox(label="Go endpoint", value=DEFAULT_ENDPOINT, scale=2)
        max_tokens = gr.Slider(16, 1024, value=256, step=16, label="Max tokens")
        student_uses_state = gr.Checkbox(value=False, label="Student uses book state")
    question = gr.Textbox(label="Student question", lines=3, placeholder="What did Marcus learn from Verus?")
    ask = gr.Button("Ask", variant="primary")
    chat = gr.Chatbot(label="Teacher/student trace", type="messages", height=420)
    with gr.Row():
        raw = gr.JSON(label="Raw go-ai response")
        state = gr.JSON(label="State")
    ask.click(ask_book_state, [endpoint, question, max_tokens, student_uses_state], [chat, raw, state])
    question.submit(ask_book_state, [endpoint, question, max_tokens, student_uses_state], [chat, raw, state])

demo.launch()